# Agentic AI in a Nutshell
**Hands-on walkthrough · Accenture Internal Knowledge Sharing · May 2026**

---

This notebook walks through every layer of an AI agent — from a bare LLM call all the way to a stateful agent that remembers who you are across conversations.

| Step | What happens |
|------|--------------|
| 1 | Plain LLM call — text in, text out |
| 2 | Define tools the LLM can use |
| 3 | Bind tools to the LLM and watch it decide |
| 4 | Full agent workflow with LangGraph |
| 5 | Human-in-the-Loop approval |
| 6 | Memory — **without** vs **with** persistence |
| Bonus | Full agent (tools + memory) |

> **Model:** `gemini-2.5-flash` · **Search:** Tavily · **Orchestration:** LangGraph

## Setup — load API keys

> **Kernel:** Select **"LangGraph Tutorial (.venv)"** in Jupyter before running.  
> All packages are already installed in `.venv/`. See README for setup steps.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()  # reads .env → GEMINI_API_KEY, TAVILY_API_KEY, LANGSMITH_*

for key in ["GEMINI_API_KEY", "TAVILY_API_KEY", "LANGSMITH_API_KEY"]:
    val = os.getenv(key, "NOT SET")
    print(f"{key}: {val[:8]}..." if val != "NOT SET" else f"{key}: NOT SET")

---
## Step 1 — Plain LLM Call

**Concept:** An LLM is just a function: text in → text out.  
No tools, no memory, no loop — a single round-trip.

```
You ──► LLM ──► Answer
```

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Instantiate Gemini 2.5 Flash
# temperature=0 → deterministic, consistent answers (good for demos)
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# .invoke() sends one message and returns one AIMessage — simplest possible call
response = llm.invoke("What is an AI agent, in one sentence?")
print(response.content)

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

# We can also pass a full conversation history as a list of messages
messages = [
    # SystemMessage sets the assistant's persona / context
    SystemMessage(content="You are a helpful assistant who explains things simply."),
    HumanMessage(content="What is the difference between a chatbot and an agent?"),
]

response = llm.invoke(messages)
print(response.content)

---
## Step 2 — Define Tools

**Concept:** Tools are Python functions the agent can call.  
The LLM never runs them directly — it *decides* to call one, and the framework executes it.

```
LLM decides ──► Agent executes tool ──► Result fed back to LLM
```

We'll define two tools:
- **`calculator`** — safe math evaluation (no internet needed)
- **`web_search`** — live Tavily web search

In [ ]:
from langchain_core.tools import tool

# @tool turns a plain Python function into a LangChain tool.
# The docstring is what the LLM reads to decide WHEN to use this tool.
@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression and return the numeric result.
    Use this whenever the user asks for a calculation.
    Example input: '(12 + 8) * 3 / 2'
    """
    try:
        # eval() scope is locked down — no builtins, no imports
        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)
    except Exception as e:
        return f"Error: {e}"


# Tools are just functions — we can test them directly
print(calculator.invoke("(12 + 8) * 3 / 2"))  # 30.0
print(calculator.invoke("2 ** 10"))             # 1024

In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults

# Pre-built Tavily tool — queries the search API and returns snippets
# max_results=3 keeps latency and cost low
web_search = TavilySearchResults(max_results=3)

results = web_search.invoke("What is LangGraph used for?")
for r in results:
    print(r["url"])
    print(r["content"][:200])
    print("---")

In [ ]:
# Bundle both tools — this list is what we hand to the LLM
tools = [calculator, web_search]

for t in tools:
    print(f"Tool : {t.name}")
    print(f"Desc : {t.description[:120]}...\n")

---
## Step 3 — LLM with Tool Calling

**Concept:** `bind_tools()` attaches tool schemas to every request.  
Now the LLM can reply with plain text **or** a tool call request.

```
User asks ──► LLM sees tool schemas
                 │
                 ├── needs tool? ──► returns tool_call
                 └── no tool? ───► returns text
```

In [ ]:
# bind_tools() sends tool schemas alongside every message so the LLM knows they exist
llm_with_tools = llm.bind_tools(tools)

response = llm_with_tools.invoke("What is 1337 multiplied by 42?")

# When the LLM wants to use a tool, .tool_calls is populated instead of .content
print("Tool calls:", response.tool_calls)
print("Text content:", response.content)

In [ ]:
from langchain_core.messages import ToolMessage

# Build a name→tool lookup so we can dispatch any tool call
tool_map = {t.name: t for t in tools}

# Conversation so far: user question + LLM's tool-call response
messages = [
    HumanMessage(content="What is 1337 multiplied by 42?"),
    response,
]

# Execute each tool the LLM requested
for tc in response.tool_calls:
    print(f"  → running '{tc['name']}' with {tc['args']}")
    result = tool_map[tc["name"]].invoke(tc["args"])
    print(f"  ← result: {result}")
    # ToolMessage wraps the result so the LLM knows which call it answers
    messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))

# Send full conversation back — LLM now has the tool result and can answer
final = llm_with_tools.invoke(messages)
print("\nFinal answer:", final.content)

---
## Step 4 — Agent Workflow with LangGraph

**Concept:** LangGraph turns the message loop into a graph of nodes and edges:

```
START ──► [agent] ──► tool needed? ──yes──► [tools] ──► back to agent
                   └──no──► END
```

Three building blocks:
- **State** — shared notebook (messages list) that flows through the graph
- **Nodes** — Python functions that read/update the state
- **Edges** — arrows that decide what node runs next

The loop runs automatically until the LLM stops requesting tools.

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

# --- State ---
# add_messages is a reducer: new messages are APPENDED, not replaced
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]


# --- Agent node ---
# Sends current messages to the LLM and appends the response
def agent_node(state: AgentState):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}


# --- Build the graph ---
builder = StateGraph(AgentState)

builder.add_node("agent", agent_node)       # LLM reasoning step
builder.add_node("tools", ToolNode(tools))  # executes whatever tool the LLM requested

builder.add_edge(START, "agent")            # always start with the agent

# tools_condition: if last message has tool_calls → "tools", else → END
builder.add_conditional_edges("agent", tools_condition)

# After tools run, go back to the agent for the next reasoning step
builder.add_edge("tools", "agent")

graph = builder.compile()
print("Graph compiled.")

In [ ]:
# ── Visualise the graph BEFORE running it ──────────────────────────────────
# This shows the full workflow as a diagram so you can see the shape first.
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    # Fallback: print Mermaid source (paste at mermaid.live to view)
    print(graph.get_graph().draw_mermaid())

In [ ]:
# ── Now run it ─────────────────────────────────────────────────────────────
print("Question: What is the latest version of LangGraph?\n")

# .stream() yields a state snapshot after each node so we can watch the loop
for step in graph.stream(
    {"messages": [HumanMessage(content="What is the latest version of LangGraph?")]},
    stream_mode="values",
):
    last = step["messages"][-1]
    kind = last.__class__.__name__
    if kind == "AIMessage" and last.tool_calls:
        for tc in last.tool_calls:
            print(f"[AGENT] → calling '{tc['name']}' with: {tc['args']}")
    elif kind == "ToolMessage":
        print(f"[TOOL]  ← {last.content[:200]}")
    elif kind == "AIMessage" and last.content:
        print(f"[AGENT] Final answer:\n{last.content}")

In [ ]:
# Ask something that uses BOTH tools in one turn
print("Question: What is 15% of 280, and who invented the calculator?\n")

for step in graph.stream(
    {"messages": [HumanMessage(content="What is 15% of 280, and who invented the calculator?")]},
    stream_mode="values",
):
    last = step["messages"][-1]
    kind = last.__class__.__name__
    if kind == "AIMessage" and last.tool_calls:
        for tc in last.tool_calls:
            print(f"[AGENT] → calling '{tc['name']}' with: {tc['args']}")
    elif kind == "ToolMessage":
        print(f"[TOOL]  ← {last.content[:200]}")
    elif kind == "AIMessage" and last.content:
        print(f"[AGENT] Final answer:\n{last.content}")

---
## Step 5 — Human-in-the-Loop

**Concept:** Some tool calls are risky (send an email, delete a record, file a ticket).  
We want a human to review before they execute.

LangGraph supports this with `interrupt_before`: the graph **pauses** before a node,  
lets a human inspect state, then **resumes** when approved.

```
agent ──► [PAUSE — human reviews]
                │
                ├── approve ──► tools ──► agent ──► END
                └── reject  ──► END
```

Requires a **checkpointer** so state survives the pause.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

hitl_checkpointer = MemorySaver()  # stores state in memory during the pause

# interrupt_before=["tools"] → pause the graph just before the tools node runs
hitl_graph = builder.compile(
    checkpointer=hitl_checkpointer,
    interrupt_before=["tools"],
)

print("Human-in-the-Loop graph compiled.")

In [ ]:
# ── Visualise HITL graph ───────────────────────────────────────────────────
try:
    display(Image(hitl_graph.get_graph().draw_mermaid_png()))
except Exception:
    print(hitl_graph.get_graph().draw_mermaid())

In [ ]:
# Each conversation thread needs a unique ID so the checkpointer can find its state
thread_cfg = {"configurable": {"thread_id": "hitl-demo-1"}}

# Run until the graph pauses at 'tools'
print("Sending question...\n")
for step in hitl_graph.stream(
    {"messages": [HumanMessage(content="Search the web: who founded Anthropic?")]},
    config=thread_cfg,
    stream_mode="values",
):
    last = step["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        print("[AGENT] Wants to call:")
        for tc in last.tool_calls:
            print(f"  Tool: {tc['name']}")
            print(f"  Args: {tc['args']}")

# Check where the graph is paused
paused_state = hitl_graph.get_state(thread_cfg)
print(f"\nGraph paused. Next node: {paused_state.next}")
print(">>> Human review required <<<")

In [ ]:
decision = input("Approve tool call? (yes/no): ").strip().lower()

if decision == "yes":
    print("\nApproved — resuming...\n")
    # Passing None as input means "continue from the pause point, no new messages"
    for step in hitl_graph.stream(None, config=thread_cfg, stream_mode="values"):
        last = step["messages"][-1]
        kind = last.__class__.__name__
        if kind == "ToolMessage":
            print(f"[TOOL]  ← {last.content[:300]}")
        elif kind == "AIMessage" and last.content:
            print(f"[AGENT] Final answer:\n{last.content}")
else:
    print("Rejected — tool call cancelled.")

---
## Step 6 — Memory: Without vs With Persistence

### Part A — Without memory (stateless)

**Concept:** Every call is independent. The agent has no idea what was said before.

```
Call 1: "My name is Fadly"  ──► [LLM]  ──► "Nice to meet you!"
Call 2: "What is my name?"  ──► [LLM]  ──► "I don't know" ← NO memory!
```

We'll prove this by sending two separate `.invoke()` calls with **no shared state**.

In [ ]:
# ── No-memory graph — plain LLM, no checkpointer ──────────────────────────
from langgraph.graph import StateGraph, START, END


def chat_node(state: AgentState):
    return {"messages": [llm.invoke(state["messages"])]}


no_memory_builder = StateGraph(AgentState)
no_memory_builder.add_node("chat", chat_node)
no_memory_builder.add_edge(START, "chat")
no_memory_builder.add_edge("chat", END)

# Compiled WITHOUT a checkpointer — each .invoke() starts completely fresh
no_memory_graph = no_memory_builder.compile()

print("No-memory graph compiled.")

In [ ]:
# ── Visualise ──────────────────────────────────────────────────────────────
try:
    display(Image(no_memory_graph.get_graph().draw_mermaid_png()))
except Exception:
    print(no_memory_graph.get_graph().draw_mermaid())

In [ ]:
# ── Invoke 1: introduce ourselves ─────────────────────────────────────────
result1 = no_memory_graph.invoke(
    {"messages": [HumanMessage(content="Hi! My name is Fadly. Remember that.")]}
)
print("[Invoke 1] Agent:", result1["messages"][-1].content)

In [ ]:
# ── Invoke 2: fresh call — does it remember? ───────────────────────────────
# This is a brand-new .invoke() with NO connection to the previous call.
result2 = no_memory_graph.invoke(
    {"messages": [HumanMessage(content="What is my name?")]}
)
print("[Invoke 2] Agent:", result2["messages"][-1].content)
print("\n👆 The agent has no idea — each call is isolated.")

### Part B — With memory (persistent)

**Concept:** A **checkpointer** saves the full message history after every node.  
As long as we use the **same `thread_id`**, the agent picks up exactly where it left off.

```
Thread A, Call 1: "My name is Fadly" ──► [saved to checkpoint]
Thread A, Call 2: "What is my name?" ──► replays history ──► "Your name is Fadly!"
Thread B, Call 1: "What is my name?" ──► fresh thread ──► "I don't know"
```

In [ ]:
memory_checkpointer = MemorySaver()  # in-memory store; swap for SQLite/Postgres in prod

# Same graph, same nodes — the ONLY difference is the checkpointer
memory_graph = no_memory_builder.compile(checkpointer=memory_checkpointer)

print("Memory graph compiled.")

In [ ]:
# ── Visualise memory graph ─────────────────────────────────────────────────
try:
    display(Image(memory_graph.get_graph().draw_mermaid_png()))
except Exception:
    print(memory_graph.get_graph().draw_mermaid())

In [ ]:
# Helper: chat with a specific thread
def chat(message: str, thread_id: str) -> str:
    config = {"configurable": {"thread_id": thread_id}}
    result = memory_graph.invoke(
        {"messages": [HumanMessage(content=message)]}, config=config
    )
    return result["messages"][-1].content


# Thread A — introduce ourselves
print("[Thread A | Turn 1]")
print("User: Hi! My name is Fadly. I work at Accenture.")
print("Agent:", chat("Hi! My name is Fadly. I work at Accenture.", "thread-A"))

In [ ]:
# Thread A — same thread, second turn — does it remember?
print("[Thread A | Turn 2]")
print("User: What is my name and where do I work?")
print("Agent:", chat("What is my name and where do I work?", "thread-A"))

In [ ]:
# Thread B — different thread_id → completely fresh state
print("[Thread B | Turn 1 — brand-new thread]")
print("User: What is my name and where do I work?")
print("Agent:", chat("What is my name and where do I work?", "thread-B"))
print("\n👆 Thread B has no history — it can't know.")

In [ ]:
# Inspect what is stored in Thread A's checkpoint
state_a = memory_graph.get_state({"configurable": {"thread_id": "thread-A"}})
print(f"Messages stored in Thread A: {len(state_a.values['messages'])}\n")
for msg in state_a.values["messages"]:
    role = msg.__class__.__name__.replace("Message", "")
    print(f"[{role}] {msg.content[:200]}")

---
## Bonus — Full Agent: Tools + Persistent Memory

Combining everything into one graph:
- Web search + calculator tools
- Persistent memory across turns

The agent can search the web, do math, and remember who you are — all in one thread.

In [ ]:
full_checkpointer = MemorySaver()

# Reuse the agent builder from Step 4 — just add a checkpointer
full_agent = builder.compile(checkpointer=full_checkpointer)

print("Full agent compiled.")

In [ ]:
# ── Visualise full agent graph ─────────────────────────────────────────────
try:
    display(Image(full_agent.get_graph().draw_mermaid_png()))
except Exception:
    print(full_agent.get_graph().draw_mermaid())

In [ ]:
THREAD = "full-demo-1"


def run(message: str):
    config = {"configurable": {"thread_id": THREAD}}
    print(f"User: {message}")
    for step in full_agent.stream(
        {"messages": [HumanMessage(content=message)]},
        config=config,
        stream_mode="values",
    ):
        last = step["messages"][-1]
        kind = last.__class__.__name__
        if kind == "AIMessage" and last.tool_calls:
            for tc in last.tool_calls:
                print(f"  [tool call] {tc['name']}({tc['args']})")
        elif kind == "ToolMessage":
            print(f"  [tool result] {last.content[:150]}")
        elif kind == "AIMessage" and last.content:
            print(f"Agent: {last.content}\n")


# Turn 1 — introduce ourselves
print("--- Turn 1 ---")
run("Hello! I'm Fadly, a software engineer. My favourite number is 42.")

In [ ]:
# Turn 2 — use search tool
print("--- Turn 2 ---")
run("What is the current weather in Jakarta?")

In [ ]:
# Turn 3 — memory check: does it still know who we are?
print("--- Turn 3 ---")
run("What is my name and what is my favourite number? You already know, don't search.")

---
## Summary

| Step | What you built | Key API |
|------|----------------|---------|
| 1 | Plain LLM call | `llm.invoke(messages)` |
| 2 | Tools | `@tool`, `TavilySearchResults` |
| 3 | Tool calling | `llm.bind_tools()`, `tool_calls` |
| 4 | LangGraph workflow | `StateGraph`, `ToolNode`, `tools_condition` |
| 5 | Human-in-the-Loop | `interrupt_before`, `get_state`, resume with `None` |
| 6A | No memory | stateless `.invoke()` — agent forgets |
| 6B | Persistent memory | `MemorySaver` + `thread_id` — agent remembers |
| Bonus | Full agent | tools + memory combined |

### Resources

- [LangGraph docs](https://langchain-ai.github.io/langgraph)
- [Building effective agents — Anthropic](https://www.anthropic.com/engineering/building-effective-agents)
- [AI Engineer Roadmap](https://roadmap.sh/ai-engineer)
- LangSmith tracing is already enabled via `LANGSMITH_TRACING=true` in your `.env`

> _Agents act, chatbots answer. The shift is loops + tools — not bigger models._